# 10 · AIA 72-hour saved-checkpoint replay

## tl;dr
**Prepared verification notebook. No scientific GPU replay has been run by preparing this file.**
The completed seed-17 model selected epoch 2. This review will restore that exact checkpoint and compare all 3,905 earlier validation predictions with the saved logits. It performs no fitting, tuning, new model selection, calibration or later-period evaluation.

## Context & Methods
Use the same frozen cases, saved training-only normalization, CNN–GRU architecture, float32 settings and batch size 16. The predeclared logit rule is `abs(replayed − saved) ≤ 1e-6 + 1e-5 × abs(saved)` for **every case**; changed or missing inputs stop the review.

### Key assumptions and limits
- This is a technical reproducibility check of serialization and inference on an already used selection block. It is not independent scientific validation or evidence of operational forecast skill.
- The model and preprocessing implementation are reused. Source-file hashes and cached image bytes are checked, but shared implementation errors may remain undetected.
- A fresh review-specific GPU handoff and separate allowance are required. The exhausted six-hour fitting ceiling remains unchanged.
- The proposed slot is at most 1,800 seconds, including loading, output and cleanup, with 60 seconds reserved for cleanup. Any failed or incomplete review stays unverified, with no automatic retry.
- No new images, environment changes, full-disk cache scans or additional GPU jobs are required.

### 1. Inspect the contract
Opening or running the notebook normally uses metadata-preview mode. Scientific execution is only through the reviewed launcher with its inherited GPU lock. The supervised runner executes plain Python cells in one process group, so it can terminate the model and loading workers together.

In [ ]:
from pathlib import Path
import json, os, sys, time
import numpy as np
import pandas as pd
import torch

if 'REPLAY_CONTEXT' not in globals():
    candidates = [Path.cwd(), *Path.cwd().parents]
    root = next(p for p in candidates if (p / 'scripts/aia72_replay.py').is_file())
    REPLAY_CONTEXT = {'mode': 'preview', 'bundle_root': str(root)}
ROOT = Path(REPLAY_CONTEXT['bundle_root']).resolve()
sys.path.insert(0, str(ROOT))
import scripts.aia72_replay as replay
from scripts.aia72_replay_contract import check_contract, verify_inputs, require
from scripts.aia72_replay import before_deadline
CONTRACT = check_contract(json.loads((ROOT / 'configs/aia72_seed17_replay_v1.json').read_text()))
display(pd.DataFrame([{'Mode': REPLAY_CONTEXT['mode'], 'Seed': CONTRACT['seed'],
                      'Selected epoch': CONTRACT['selected_epoch'], 'Cases': CONTRACT['case_count'],
                      'Maximum slot (minutes)': CONTRACT['slot_seconds'] / 60,
                      'Fitting allowed': CONTRACT['fitting_allowed']}]))

## Data
### 2. Verify pinned metadata and exact support
This preview reads local metadata and checkpoint hashes only. It does not read cached images or initialize CUDA. Saved normalization is reused; no statistics are fitted.

In [ ]:
source_root = REPLAY_CONTEXT.get('preview_source_root') if REPLAY_CONTEXT['mode'] == 'preview' else None
paths = verify_inputs(CONTRACT, source_root)
selected, reference, records, normalization = replay.load_support(CONTRACT, paths)
display(pd.DataFrame([{'Frozen validation cases': len(selected), 'Positive windows': int(selected.label.sum()),
                      'Distinct image objects': len(records), 'Saved normalization fit role': normalization['fit_role'],
                      'Image pixels read in this cell': 0}]))

### 3. Inspect the restored architecture
These classes are unchanged from the completed fit. The selected weight file is loaded only during authorized replay.

In [ ]:
"""AIA CNN-GRU architecture reused from Watchman77/solar-flare-aia-training.

The three classes are preserved from 19A2_Temporal_AIA_CNN_GRU_Cycle24_Training.ipynb.
Only the module wrapper is new. No upstream weights or normalization are reused.
The new 72-hour label/role contract is supplied by this repository's loader.
Provenance: results/aia72_canary_20261002/source_provenance.json.
"""

import torch
from torch import nn

DROPOUT = 0.30

class ConvBlock(nn.Module):
    def __init__(self, cin, cout, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.net(x)

class FrameCNN(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(6, 32, 0.05),
            ConvBlock(32, 64, 0.05),
            ConvBlock(64, 128, 0.10),
            ConvBlock(128, 192, 0.10),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(192, embed_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.proj(self.encoder(x))

class TemporalAIACNNGRU(nn.Module):
    def __init__(self, embed_dim=256, hidden_dim=192):
        super().__init__()
        self.frame_encoder = FrameCNN(embed_dim)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        b,t,c,h,w = x.shape
        z = self.frame_encoder(x.reshape(b*t,c,h,w)).reshape(b,t,-1)
        _, hlast = self.gru(z)
        return self.head(hlast[-1]).squeeze(-1)


### 4. Inspect the inference loop
Evaluation mode disables dropout and running-statistic updates. The loop checks all IDs, labels, finite logits and unchanged model parameters/buffers. There is no optimizer or training call.

In [ ]:
def infer_rows(model, batches, expected, device, deadline, progress=None, guard=None):
    """Independent inference loop: fixed order, no optimizer, unchanged model buffers."""
    model.eval()
    original = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    rows = []
    if guard:
        guard()
    with torch.no_grad():
        for images, labels, identities in batches:
            before_deadline(deadline)
            if guard:
                guard()
            offset = len(rows)
            require(list(identities) == expected.forecast_case_id.iloc[offset:offset + len(identities)].tolist(),
                    'Loader omitted/reordered/duplicated cases')
            require(np.array_equal(labels.numpy(), expected.label.iloc[offset:offset + len(labels)].to_numpy()),
                    'Loader outcomes changed')
            values = model(images.to(device, non_blocking=True)).detach().cpu().numpy()
            require(values.shape == (len(identities),) and np.isfinite(values).all(), 'Invalid replay logits')
            rows.extend({'forecast_case_id': identity, 'label': int(label), 'logit': float(value)}
                        for identity, label, value in zip(identities, labels, values))
            if progress:
                progress(len(rows))
            before_deadline(deadline)
            if guard:
                guard()
    require(len(rows) == len(expected), 'Incomplete replay support')
    require(all(torch.equal(original[k], v.detach().cpu()) for k, v in model.state_dict().items()),
            'Inference mutated parameters or buffers')
    return pd.DataFrame(rows)

replay.infer_rows = infer_rows
replay.TemporalAIACNNGRU = TemporalAIACNNGRU

## Results
### 5. Run only with the approved launcher
The launcher checks the fresh allowance and handoff before starting this process. Preview mode reports preparation status without generating predictions.

In [ ]:
if REPLAY_CONTEXT['mode'] == 'authorized_gpu_replay':
    import fcntl
    from scripts.aia72_replay_contract import LOCK_INODE
    require(json.loads(os.environ['GRAYBOX_REPLAY_CONTEXT']) == REPLAY_CONTEXT, 'Launcher context missing')
    lease_fd = int(os.environ['GRAYBOX_REPLAY_LEASE_FD'])
    require(os.fstat(lease_fd).st_ino == LOCK_INODE, 'Shared lease missing')
    fcntl.flock(lease_fd, fcntl.LOCK_EX | fcntl.LOCK_NB)
    result = replay.run_replay(CONTRACT, REPLAY_CONTEXT['output'], REPLAY_CONTEXT['deadline'], REPLAY_CONTEXT['owner_sha256'])
else:
    require(REPLAY_CONTEXT['mode'] == 'preview', 'Unknown execution mode')
    result = {'status': 'prepared_only_no_GPU_replay', 'cases_replayed': 0,
              'scientific_acceptance': False, 'fitting_steps': 0}
shown = ['status', 'cases', 'cases_replayed', 'mismatched_cases', 'max_absolute_logit_difference',
         'saved_log_loss', 'replayed_log_loss', 'fitting_steps', 'scientific_acceptance']
display(pd.DataFrame([{'Check': name, 'Value': result[name]} for name in shown if name in result]))

## Takeaways
### 6. Read the execution status literally
A metadata preview or successful synthetic test does not verify the scientific model. A completed comparison is still pending the controller's source recheck, process cleanup and final receipt. Keep the prediction comparison, executed notebook and controller receipt together. Later-period validation and completion of seeds 29 and 43 remain separate work.

In [ ]:
print('Notebook status:', result['status'])
print('Scientific acceptance: False. No new fitting or later-period evaluation.')